# YOLO Object Detection Training

This notebook trains a YOLO model for parking space detection using the prepared YOLO dataset.

Features:
- Train YOLOv8 or YOLOv11 model
- Monitor training metrics (mAP, precision, recall)
- Save best model weights
- Run inference on images and videos


## 1. Setup and Configuration


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
YOLO_ROOT = BASE_DIR / "data" / "processed" / "yolo_parking"
DATA_YAML = YOLO_ROOT / "data.yaml"
RUNS_DIR = BASE_DIR / "runs"

print("BASE_DIR:", BASE_DIR)
print("DATA_YAML:", DATA_YAML, "exists:", DATA_YAML.exists())
print("YOLO_ROOT:", YOLO_ROOT, "exists:", YOLO_ROOT.exists())

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if DATA_YAML.exists():
    print("\ndata.yaml contents:")
    print("="*60)
    print(DATA_YAML.read_text())
else:
    print("\nERROR: data.yaml not found!")
    print("Please run notebook 04_yolo_dataset_prep.ipynb first to create the dataset.")


BASE_DIR: c:\Harosha\George Brown\DL2\parking-vision
DATA_YAML: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml exists: True
YOLO_ROOT: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking exists: True
Using device: cpu

data.yaml contents:
path: c:/Harosha/George Brown/DL2/parking-vision/data/processed/yolo_parking

train: images/train
val: images/val

names:
  0: free_parking_space
  1: not_free_parking_space
  2: partially_free_parking_space



## 2. Load Pretrained Model


In [ ]:
# Choose model size: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (xlarge)
# For faster training and inference, start with yolov8n
# For better accuracy, use yolov8s or yolov8m

MODEL_SIZE = "n"  # Options: "n", "s", "m", "l", "x"
MODEL_NAME = f"yolov8{MODEL_SIZE}.pt"

print(f"Loading pretrained model: {MODEL_NAME}")
model = YOLO(MODEL_NAME)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model will be fine-tuned on parking space detection dataset")


Loading pretrained model: yolov8n.pt
Model loaded: yolov8n.pt
Model will be fine-tuned on parking space detection dataset


## 3. Training Configuration


In [ ]:
# Training hyperparameters
TRAIN_CONFIG = {
    "data": str(DATA_YAML),
    "epochs": 50,              # Number of training epochs
    "imgsz": 640,              # Image size (640 is standard for YOLO)
    "batch": 16,               # Batch size (adjust based on GPU memory)
    "name": "yolo_parking_v1", # Experiment name
    "project": str(RUNS_DIR),  # Project directory
    "patience": 10,            # Early stopping patience
    "save": True,              # Save checkpoints
    "save_period": 10,         # Save checkpoint every N epochs
    "val": True,               # Validate during training
    "plots": True,             # Generate training plots
    "device": device,          # Device to use
}

print("Training configuration:")
print("="*60)
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")
print("="*60)


Training configuration:
  data: c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml
  epochs: 50
  imgsz: 640
  batch: 16
  name: yolo_parking_v1
  project: c:\Harosha\George Brown\DL2\parking-vision\runs
  patience: 10
  save: True
  save_period: 10
  val: True
  plots: True
  device: cpu


## 4. Train Model


In [4]:


results = model.train(**TRAIN_CONFIG)


Ultralytics 8.3.228  Python-3.12.4 torch-2.5.1+cu121 CPU (AMD Ryzen 7 7730U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Harosha\George Brown\DL2\parking-vision\data\processed\yolo_parking\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo_parking_v1, nbs=64, nms=False, opset=None, optimize=False, optimize

## 5. Training Results Summary


In [ ]:
# Training results are saved in runs/detect/yolo_parking_v1/
results_dir = RUNS_DIR / "detect" / TRAIN_CONFIG["name"]
best_model_path = results_dir / "weights" / "best.pt"
last_model_path = results_dir / "weights" / "last.pt"

print("Training results:")
print("="*60)
print(f"Results directory: {results_dir}")
print(f"Best model: {best_model_path}")
print(f"Last model: {last_model_path}")

if best_model_path.exists():
    print(f"\n✓ Best model saved: {best_model_path}")
    file_size_mb = best_model_path.stat().st_size / (1024 * 1024)
    print(f"  File size: {file_size_mb:.2f} MB")
else:
    print("\n⚠ Best model not found!")

# Display key metrics if available
if hasattr(results, 'results_dict'):
    print("\nFinal metrics:")
    for key, value in results.results_dict.items():
        if isinstance(value, (int, float)):
            print(f"  {key}: {value:.4f}")


<>:28: SyntaxWarning: invalid escape sequence '\ '
<>:28: SyntaxWarning: invalid escape sequence '\ '


C:\Users\haros\AppData\Local\Temp\ipykernel_7560\3206923458.py:28: SyntaxWarning: invalid escape sequence '\ '
  print("\ Best model not found!")


Training results:
Results directory: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1
Best model: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\best.pt
Last model: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\last.pt

 Best model saved: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\best.pt
  File size: 5.96 MB

 Final metrics:
  metrics/precision(B): 0.9050
  metrics/recall(B): 0.4706
  metrics/mAP50(B): 0.5429
  metrics/mAP50-95(B): 0.4320
  fitness: 0.4320


## 6. Load Best Model for Inference


In [ ]:
# Load the best model
if best_model_path.exists():
    best_model = YOLO(str(best_model_path))
    print(f"Loaded best model from: {best_model_path}")
    print("Model ready for inference!")
else:
    print(f"ERROR: Best model not found at {best_model_path}")
    print("Please check if training completed successfully.")
    best_model = None


Loaded best model from: c:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_v1\weights\best.pt


## 7. Inference on Validation Images


In [ ]:
if best_model is not None:
    # Get validation images
    from glob import glob
    
    val_images_dir = YOLO_ROOT / "images" / "val"
    val_images = glob(str(val_images_dir / "*.jpg")) + glob(str(val_images_dir / "*.png"))
    
    print(f"Found {len(val_images)} validation images")
    
    if val_images:
        # Run inference on first 5 validation images
        num_test_images = min(5, len(val_images))
        test_images = val_images[:num_test_images]
        
        print(f"\nRunning inference on {num_test_images} validation images...")
        
        results = best_model.predict(
            source=test_images,
            save=True,
            project=str(RUNS_DIR),
            name="yolo_parking_pred_images",
            conf=0.25,        # Confidence threshold
            imgsz=640,       # Image size
            show_labels=True,
            show_conf=True,
        )
        
        print(f"\n✓ Inference complete!")
        print(f"Annotated images saved to: {RUNS_DIR / 'detect' / 'yolo_parking_pred_images'}")
        
        # Display results summary
        if results:
            print(f"\nProcessed {len(results)} images")
            for i, result in enumerate(results[:3]):  # Show first 3
                print(f"  Image {i+1}: {len(result.boxes)} detections")
    else:
        print("No validation images found!")
else:
    print("Model not loaded. Please train the model first.")


Found 6 validation images

Running inference on 5 validation images...

0: 640x640 11 not_free_parking_spaces, 123.5ms
1: 640x640 1 free_parking_space, 25 not_free_parking_spaces, 123.5ms
2: 640x640 6 free_parking_spaces, 21 not_free_parking_spaces, 123.5ms
3: 640x640 27 free_parking_spaces, 36 not_free_parking_spaces, 123.5ms
4: 640x640 13 free_parking_spaces, 11 not_free_parking_spaces, 123.5ms
Speed: 12.3ms preprocess, 123.5ms inference, 9.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to C:\Harosha\George Brown\DL2\parking-vision\runs\yolo_parking_pred_images

 Inference complete!
Annotated images saved to: c:\Harosha\George Brown\DL2\parking-vision\runs\detect\yolo_parking_pred_images

Processed 5 images
  Image 1: 11 detections
  Image 2: 26 detections
  Image 3: 27 detections


## 8. Inference on Video


In [ ]:
if best_model is not None:
    # Video paths
    VIDEO_DIR = BASE_DIR / "data" / "raw" / "UFPARK - Dataset" / "morning_samples_anonimized"
    video_files = list(VIDEO_DIR.glob("*.avi")) + list(VIDEO_DIR.glob("*.mp4"))
    
    if video_files:
        # Select first video (or change index)
        SELECTED_VIDEO_INDEX = 0
        VIDEO_INPUT = video_files[SELECTED_VIDEO_INDEX]
        
        print(f"Selected video: {VIDEO_INPUT.name}")
        print(f"Running YOLO inference on video...")
        print("This may take a while depending on video length.")
        print("="*60)
        
        results = best_model.predict(
            source=str(VIDEO_INPUT),
            save=True,
            project=str(RUNS_DIR),
            name="yolo_parking_video_v1",
            conf=0.25,        # Confidence threshold
            imgsz=640,       # Image size
            show_labels=True,
            show_conf=True,
        )
        
        # Output video path
        output_video_dir = RUNS_DIR / "detect" / "yolo_parking_video_v1"
        output_video = output_video_dir / VIDEO_INPUT.name
        
        print(f"\n✓ Video inference complete!")
        print(f"Annotated video saved to: {output_video}")
        
        if output_video.exists():
            file_size_mb = output_video.stat().st_size / (1024 * 1024)
            print(f"Output file size: {file_size_mb:.2f} MB")
    else:
        print(f"No video files found in {VIDEO_DIR}")
        print("Please check the video directory path.")
else:
    print("Model not loaded. Please train the model first.")


Selected video: 2020-03-19-08-00-00.scene_0000TESTE_BB_write.avi

WARNING 
inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/599) c:\Harosha\George Brown\DL2\parking-vision\data\raw\UFPARK - Dataset\morning_samples_anonimized\2020-03-19-08-00-00.scene_0000TESTE_BB_write.avi: 480x640 6 free_parking_spaces, 6 not_free_parking_spaces, 145.9ms
video 1/1 (frame 2/599) c:\Harosha\George Brown\DL2\parking-vision\data\raw\UFPARK - Dataset\morning_samples_anonimized\2020-03-19-08-00-00.scene_0000TESTE_BB

In [ ]:
# Set to True to process all videos
PROCESS_ALL_VIDEOS = False

if PROCESS_ALL_VIDEOS and best_model is not None:
    VIDEO_DIR = BASE_DIR / "data" / "raw" / "UFPARK - Dataset" / "morning_samples_anonimized"
    video_files = list(VIDEO_DIR.glob("*.avi")) + list(VIDEO_DIR.glob("*.mp4"))
    video_files.sort()
    
    print(f"Processing {len(video_files)} videos...")
    print("="*60)
    
    for i, video_path in enumerate(video_files):
        print(f"\n[{i+1}/{len(video_files)}] Processing: {video_path.name}")
        
        results = best_model.predict(
            source=str(video_path),
            save=True,
            project=str(RUNS_DIR),
            name=f"yolo_parking_video_batch",
            conf=0.25,
            imgsz=640,
            show_labels=True,
            show_conf=True,
        )
        
        output_video_dir = RUNS_DIR / "detect" / "yolo_parking_video_batch"
        output_video = output_video_dir / video_path.name
        
        if output_video.exists():
            file_size_mb = output_video.stat().st_size / (1024 * 1024)
            print(f"  ✓ Saved: {output_video.name} ({file_size_mb:.2f} MB)")
    
    print(f"\n{'='*60}")
    print(f"Batch processing complete! Processed {len(video_files)} videos.")
    print(f"Output directory: {RUNS_DIR / 'detect' / 'yolo_parking_video_batch'}")
else:
    if not PROCESS_ALL_VIDEOS:
        print("Batch processing is disabled. Set PROCESS_ALL_VIDEOS = True to enable.")
    elif best_model is None:
        print("Model not loaded. Please train the model first.")


## 10. Summary

The YOLO object detection pipeline is now complete:

1. **Dataset Preparation**: Converted HuggingFace XML annotations to YOLO format
2. **Model Training**: Trained YOLOv8 model on parking space detection
3. **Inference**: Tested on validation images and videos

**Key Files:**
- Best model: `runs/detect/yolo_parking_v1/weights/best.pt`
- Training plots: `runs/detect/yolo_parking_v1/results.png`
- Annotated images: `runs/detect/yolo_parking_pred_images/`
- Annotated videos: `runs/detect/yolo_parking_video_v1/`

**Next Steps:**
- Evaluate model performance on test set
- Fine-tune hyperparameters if needed
- Compare with CNN baseline results
- Deploy model for real-time inference
